In [31]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')


In [32]:
PALETTE   = sns.color_palette("tab10")
BLUE      = "#2563EB"
RED       = "#DC2626"
GREEN     = "#16A34A"
ORANGE    = "#D97706"
GRAY      = "#6B7280"
sns.set_theme(style="whitegrid", font_scale=1.05)

In [33]:
df = pd.read_csv(r"D:\Data Analytics\Project 5\Project 5.1\occupazione.csv")
year_cols = [str(y) for y in range(2015, 2025)]

print(f"\nShape      : {df.shape}")
print(f"Columns    : {df.columns.tolist()}")
print(f"\nSEX values : {sorted(df['SEX'].unique().tolist())}")
print(f"AGE groups : {sorted(df['AGE'].unique().tolist())}")
print(f"Countries  : {len(df['ISO'].unique())} — {sorted(df['ISO'].unique().tolist())}")
print("\nFirst 5 rows:")
print(df.head())


Shape      : (420, 13)
Columns    : ['SEX', 'AGE', 'ISO', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

SEX values : ['F', 'M']
AGE groups : ['15-24', '15-29', '15-64', '20-64', '25-54', '55-64']
Countries  : 35 — ['AT', 'BA', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GR', 'HR', 'HU', 'IE', 'IS', 'IT', 'LT', 'LU', 'LV', 'ME', 'MK', 'MT', 'NL', 'NO', 'PL', 'PT', 'RO', 'RS', 'SE', 'SI', 'SK', 'TR']

First 5 rows:
  SEX    AGE ISO  2015  2016  2017  2018  2019  2020  2021  2022  2023  2024
0   F  15-24  AT  48.7  49.0  49.0  48.7  48.4  47.1  45.7  48.2  49.8  48.3
1   F  15-24  BA   NaN   NaN   NaN   NaN   NaN   NaN  13.7  12.3  11.5  10.4
2   F  15-24  BE  21.7  21.4  20.9  23.5  25.8  22.3  23.7  25.4  25.9  25.2
3   F  15-24  BG  14.0  13.8  16.3  14.5  15.6  13.4  13.0  15.9  16.2  13.7
4   F  15-24  CH  62.6  63.2  62.4  62.6  61.6  59.4  59.5  60.2  59.9  59.1


In [34]:
# ── 2a. Missing value audit ──────────────────────────────────
print("\n[NaN per year column]")
print(df[year_cols].isnull().sum())

nan_iso = (df.set_index(['SEX', 'AGE', 'ISO'])[year_cols]
             .isnull().any(axis=1)
             .groupby('ISO').sum())
print("\n[Countries with any missing data]")
print(nan_iso[nan_iso > 0])


[NaN per year column]
2015    12
2016    12
2017    12
2018    12
2019    12
2020    12
2021    12
2022    12
2023    12
2024    12
dtype: int64

[Countries with any missing data]
ISO
BA    12
ME    12
dtype: int64


In [35]:
# ── 2b. Convert to long (tidy) format ───────────────────────
df_long = (df
    .melt(id_vars=['SEX', 'AGE', 'ISO'],
          value_vars=year_cols,
          var_name='Year',
          value_name='Rate')
    .assign(Year=lambda x: x['Year'].astype(int),
            Rate=lambda x: pd.to_numeric(x['Rate'], errors='coerce'))
    .rename(columns=str.upper)
)

In [36]:
# Standardise column capitalisation for convenience
df_long.columns = ['SEX', 'AGE', 'ISO', 'YEAR', 'RATE']

print(f"\nLong-format shape : {df_long.shape}")
print(f"Total NaN in RATE : {df_long['RATE'].isna().sum()} "
      f"({df_long['RATE'].isna().mean()*100:.1f}%)")


Long-format shape : (4200, 5)
Total NaN in RATE : 120 (2.9%)


In [37]:
# ── 2c. Duplicate check ──────────────────────────────────────
dupes = df_long.duplicated(subset=['SEX', 'AGE', 'ISO', 'YEAR']).sum()
print(f"Duplicate rows    : {dupes}")

Duplicate rows    : 0


In [38]:
# ── 2d. Helper filters ───────────────────────────────────────
def age_filter(age: str):
    return df_long[df_long['AGE'] == age]

emp_1564  = age_filter('15-64')   # Headline employment metric
emp_1524  = age_filter('15-24')   # Youth
emp_5564  = age_filter('55-64')   # Senior
emp_2554  = age_filter('25-54')   # Prime age

In [39]:
# ── 3a. Descriptive statistics ───────────────────────────────
print("\n[Employment rate stats — all age groups]")
print(df_long.groupby('AGE')['RATE'].describe().round(2))
print("\n[Employment rate stats — by gender (15-64)]")
print(emp_1564.groupby('SEX')['RATE'].describe().round(2))


[Employment rate stats — all age groups]
       count   mean    std   min    25%    50%    75%   max
AGE                                                        
15-24  680.0  34.67  15.71  10.4  23.48  30.65  45.60  78.2
15-29  680.0  49.35  12.95  23.0  40.05  48.75  57.72  82.5
15-64  680.0  68.31  10.30  29.7  64.00  70.00  75.00  89.4
20-64  680.0  72.78  10.39  32.0  68.90  74.90  79.20  91.1
25-54  680.0  80.18   9.67  35.7  77.50  82.10  86.30  94.8
55-64  680.0  58.20  14.76  16.7  48.72  60.25  69.30  89.7

[Employment rate stats — by gender (15-64)]
     count   mean    std   min    25%   50%    75%   max
SEX                                                     
F    340.0  63.35  11.02  29.7  57.85  66.3  71.03  83.6
M    340.0  73.26   6.48  56.0  69.00  73.0  77.90  89.4


In [40]:
# ── 3b. Year-over-year trend (15-64) ────────────────────────
yoy = emp_1564.groupby('YEAR')['RATE'].mean()
yoy_chg = yoy.diff().rename('YoY Change')
yoy_df = pd.DataFrame({'Mean Rate': yoy.round(2), 'YoY Change': yoy_chg.round(2)})
print("\n[YoY employment trend (15-64 aggregate)]")
print(yoy_df)


[YoY employment trend (15-64 aggregate)]
      Mean Rate  YoY Change
YEAR                       
2015      64.51         NaN
2016      65.57        1.06
2017      66.89        1.32
2018      68.07        1.18
2019      68.88        0.81
2020      67.70       -1.18
2021      68.77        1.08
2022      70.46        1.69
2023      70.88        0.42
2024      71.34        0.46


In [41]:
# ── 3c. COVID shock quantification ──────────────────────────
covid_pivot = (emp_1564
    .groupby(['ISO', 'YEAR'])['RATE'].mean()
    .unstack())
covid_drop = (covid_pivot[2020] - covid_pivot[2019]).dropna().sort_values()
print("\n[Top 8 COVID-impacted countries — 2019→2020 drop (pp)]")
print(covid_drop.head(8).round(2))
print("\n[Most resilient countries — 2019→2020]")
print(covid_drop.tail(5).round(2))


[Top 8 COVID-impacted countries — 2019→2020 drop (pp)]
ISO
ME   -5.75
IS   -3.85
IE   -2.90
TR   -2.80
GR   -2.45
ES   -2.40
AT   -1.90
SE   -1.80
dtype: float64

[Most resilient countries — 2019→2020]
ISO
MK   -0.05
HR    0.20
PL    0.20
MT    0.50
RS    0.70
dtype: float64


In [42]:
# ── 3d. Recovery: 2020→2024 growth ──────────────────────────
recovery = (covid_pivot[2024] - covid_pivot[2020]).dropna().sort_values(ascending=False)
print("\n[Recovery gain 2020→2024 (pp)]")
print(recovery.head(8).round(2))


[Recovery gain 2020→2024 (pp)]
ISO
GR    9.55
IE    7.95
TR    7.60
RS    7.20
MT    6.80
HR    6.05
ES    5.15
IS    5.00
dtype: float64


In [43]:
# ── 3e. Overall growth 2015→2024 ────────────────────────────
total_growth = (covid_pivot[2024] - covid_pivot[2015]).dropna().sort_values(ascending=False)
print("\n[Total growth 2015→2024 (pp)]")
print(total_growth.round(2))


[Total growth 2015→2024 (pp)]
ISO
RS    15.25
MT    13.75
CY    12.85
GR    12.50
HR    12.25
PL    10.90
MK    10.10
PT    10.00
IE     9.70
HU     9.15
RO     8.85
BG     8.80
SI     8.45
ES     8.25
SK     7.90
NL     6.70
LT     6.35
IT     6.15
DK     5.25
CZ     5.15
FI     5.05
BE     5.00
TR     4.95
DE     4.35
FR     4.25
EE     3.60
LU     3.60
LV     3.05
AT     3.00
NO     1.60
SE     1.50
CH     1.20
IS     0.50
dtype: float64


In [44]:
# ── 3f. Senior employment surge (55-64) ─────────────────────
print("\n[Senior (55-64) employment trend]")
print(emp_5564.groupby('YEAR')['RATE'].mean().round(2))


[Senior (55-64) employment trend]
YEAR
2015    51.11
2016    53.04
2017    55.00
2018    56.85
2019    58.28
2020    58.50
2021    59.84
2022    61.80
2023    63.16
2024    64.38
Name: RATE, dtype: float64


In [45]:
gender_trend = (emp_1564
    .groupby(['YEAR', 'SEX'])['RATE'].mean()
    .unstack()
    .assign(gap=lambda x: x['M'] - x['F']))
print("\n[Gender employment rates and M-F gap (15-64)]")
print(gender_trend.round(2))

gap_2024 = (emp_1564[emp_1564['YEAR'] == 2024]
    .groupby(['ISO', 'SEX'])['RATE'].mean()
    .unstack()
    .assign(gap=lambda x: (x['M'] - x['F']).round(2))
    .sort_values('gap', ascending=False))
print("\n[Gender gap by country (2024)]")
print(gap_2024.round(2))


[Gender employment rates and M-F gap (15-64)]
SEX       F      M    gap
YEAR                     
2015  59.41  69.61  10.20
2016  60.48  70.66  10.18
2017  61.79  71.99  10.19
2018  62.94  73.20  10.26
2019  63.81  73.95  10.14
2020  62.76  72.63   9.87
2021  63.70  73.85  10.15
2022  65.60  75.32   9.73
2023  66.22  75.53   9.31
2024  66.81  75.87   9.05

[Gender gap by country (2024)]
SEX     F     M   gap
ISO                  
TR   36.9  73.2  36.3
BA   41.0  66.5  25.5
IT   53.3  71.1  17.8
GR   54.6  72.0  17.4
RO   55.3  72.0  16.7
MK   49.6  66.0  16.4
MT   71.9  85.1  13.2
CZ   69.5  81.1  11.6
RS   60.8  71.7  10.9
PL   67.2  77.8  10.6
CY   71.1  80.3   9.2
ES   61.6  70.5   8.9
SK   68.3  76.5   8.2
IE   70.6  78.4   7.8
HU   71.4  78.7   7.3
BE   63.3  70.2   6.9
AT   70.7  77.5   6.8
NL   78.9  85.7   6.8
CH   77.0  83.7   6.7
DE   74.1  80.7   6.6
BG   67.6  74.1   6.5
SI   70.0  76.0   6.0
IS   82.3  88.1   5.8
HR   65.4  71.1   5.7
DK   74.5  79.9   5.4
FR   66.4  71.6

In [46]:
# Correlation M↔F
corr_mf = gap_2024['M'].corr(gap_2024['F'])
print(f"\nCorrelation M vs F employment (2024): {corr_mf:.3f}")


Correlation M vs F employment (2024): 0.731


In [47]:
# ==============================================================
# 5. COUNTRY RANKING (2024, 15-64)
# ==============================================================
rank_2024 = (emp_1564[emp_1564['YEAR'] == 2024]
    .groupby('ISO')['RATE'].mean()
    .dropna()
    .sort_values(ascending=False))
print("\n[Full country ranking — mean employment rate 2024]")
print(rank_2024.round(2))

print(f"\nTOP 5    : {rank_2024.head(5).index.tolist()}")
print(f"BOTTOM 5 : {rank_2024.tail(5).index.tolist()}")


[Full country ranking — mean employment rate 2024]
ISO
IS    85.20
NL    82.30
CH    80.35
MT    78.50
DE    77.40
DK    77.20
NO    77.00
SE    76.65
EE    75.70
CY    75.70
CZ    75.30
HU    75.05
IE    74.50
AT    74.10
LT    73.60
SI    73.00
PT    72.85
FI    72.60
PL    72.50
SK    72.40
LV    71.15
BG    70.85
LU    69.65
FR    69.00
HR    68.25
BE    66.75
RS    66.25
ES    66.05
RO    63.65
GR    63.30
IT    62.20
MK    57.80
TR    55.05
BA    53.75
Name: RATE, dtype: float64

TOP 5    : ['IS', 'NL', 'CH', 'MT', 'DE']
BOTTOM 5 : ['GR', 'IT', 'MK', 'TR', 'BA']


In [48]:
# ==============================================================
# 6. AGE-GROUP DEEP DIVE
# ==============================================================
youth_trend = (emp_1524
    .groupby(['YEAR', 'SEX'])['RATE'].mean()
    .unstack())
print("\n[Youth (15-24) employment by year & gender]")
print(youth_trend.round(2))

prime_2024 = (emp_2554[emp_2554['YEAR'] == 2024]
    .groupby('ISO')['RATE'].mean()
    .dropna().sort_values(ascending=False))
print("\n[Prime age (25-54) employment ranking — 2024]")
print(prime_2024.round(2))



[Youth (15-24) employment by year & gender]
SEX       F      M
YEAR              
2015  31.17  34.71
2016  31.79  35.61
2017  32.60  36.63
2018  33.43  37.35
2019  33.40  37.63
2020  30.50  35.10
2021  31.46  36.28
2022  33.58  37.96
2023  33.94  38.23
2024  33.70  38.33

[Prime age (25-54) employment ranking — 2024]
ISO
SI    89.70
IS    89.20
MT    88.40
HU    88.15
CZ    87.75
CH    87.00
NL    86.90
PT    86.70
PL    86.50
SE    86.10
EE    85.70
SK    85.70
AT    85.40
LT    85.25
DE    85.25
CY    84.95
LU    84.95
HR    84.60
IE    84.45
BG    84.00
DK    83.65
NO    83.65
FR    83.10
LV    81.75
BE    81.40
FI    81.20
RS    79.60
ES    78.65
RO    78.25
GR    77.15
IT    74.50
MK    70.45
BA    67.75
TR    64.15
Name: RATE, dtype: float64


In [49]:
output_path="D:\Data Analytics\Project 5\Project 5.1\Output"


In [57]:
import os
import matplotlib.pyplot as plt

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"

# Create folder if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# File path
file_path = os.path.join(output_dir, "fig1_headline_trend.png")

# ==============================
# 2. CREATE PLOT
# ==============================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# LEFT PLOT (example - modify as needed)
ax = axes[0]
headline = emp_1564.groupby("YEAR")["RATE"].mean()
ax.plot(headline.index, headline.values, marker='o')
ax.set_title("Headline Employment Rate (15-64)")
ax.set_xlabel("Year")
ax.set_ylabel("Employment Rate (%)")

# RIGHT PLOT (Gender split)
ax = axes[1]

for sex, color, label in [('F', 'red', 'Female'), ('M', 'blue', 'Male')]:
    data = emp_1564.groupby(['YEAR', 'SEX'])['RATE'].mean().unstack()[sex]
    ax.plot(data.index, data.values, marker='o', label=label, color=color)

ax.axvspan(2019.5, 2020.5, color='gray', alpha=0.12, label='COVID-19')

ax.set_title("Employment Rate by Gender (15-64)\n2015–2024", fontweight='bold')
ax.set_xlabel("Year")
ax.set_ylabel("Employment Rate (%)")
ax.legend()

# ==============================
# 3. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')  # Save first
plt.show()  # Then display
plt.close()  # Then close

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig1_headline_trend.png


In [58]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "fig2_age_trends.png")

# ==============================
# 2. CREATE PLOT
# ==============================
fig, ax = plt.subplots(figsize=(11, 6))

age_order = ['15-24', '15-29', '25-54', '55-64', '15-64', '20-64']
colors = sns.color_palette("tab10", len(age_order))

for age, color in zip(age_order, colors):
    s = df_long[df_long['AGE'] == age].groupby('YEAR')['RATE'].mean()
    ax.plot(s.index, s.values, lw=2.2, marker='o', markersize=4,
            label=age, color=color)

# Highlight COVID period
ax.axvspan(2019.5, 2020.5, color='red', alpha=0.1, label='COVID-19')

# Labels & Title
ax.set_title("Employment Rate by Age Group — 2015–2024",
             fontweight='bold', fontsize=14)
ax.set_xlabel("Year")
ax.set_ylabel("Employment Rate (%)")

# Legend outside
ax.legend(title="Age Group", bbox_to_anchor=(1.01, 1), loc='upper left')

# ==============================
# 3. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig2_age_trends.png


In [59]:
import os
import matplotlib.pyplot as plt

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "fig3_country_ranking.png")

# ==============================
# 2. CREATE PLOT
# ==============================
fig, ax = plt.subplots(figsize=(12, 8))

colors_bar = [
    GREEN if r >= 72 else (ORANGE if r >= 62 else RED)
    for r in rank_2024.values
]

ax.barh(
    rank_2024.index[::-1],
    rank_2024.values[::-1],
    color=colors_bar[::-1],
    edgecolor='white'
)

# Mean line
mean_val = rank_2024.mean()
ax.axvline(mean_val, color='navy', lw=1.5, linestyle='--',
           label=f'Mean = {mean_val:.1f}%')

# Labels & Title
ax.set_title("Employment Rate by Country — 2024 (15-64, avg M+F)",
             fontweight='bold', fontsize=13)
ax.set_xlabel("Employment Rate (%)")

ax.legend()

# ==============================
# 3. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig3_country_ranking.png


In [60]:
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "fig4_gender_gap_heatmap.png")

# ==============================
# 2. PREPARE DATA
# ==============================
gap_matrix = (
    emp_1564
    .groupby(['ISO', 'YEAR', 'SEX'])['RATE'].mean()
    .unstack('SEX')
    .assign(gap=lambda x: x['M'] - x['F'])['gap']
    .unstack('YEAR')
    .dropna(how='all')
)

# ==============================
# 3. CREATE HEATMAP
# ==============================
fig, ax = plt.subplots(figsize=(13, 9))

sns.heatmap(
    gap_matrix,
    annot=True,
    fmt=".1f",
    cmap="RdYlGn_r",
    linewidths=0.4,
    ax=ax,
    cbar_kws={'label': 'M − F gap (pp)'}
)

# Labels & Title
ax.set_title(
    "Gender Employment Gap (M − F) by Country & Year\n(15-64)",
    fontweight='bold',
    fontsize=13
)
ax.set_xlabel("Year")
ax.set_ylabel("Country")

# ==============================
# 4. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig4_gender_gap_heatmap.png


In [61]:
# ── FIG 5: COVID shock bar chart ─────────────────────────────
import os
import matplotlib.pyplot as plt

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "fig5_covid_shock.png")

# ==============================
# 2. CREATE PLOT
# ==============================
fig, ax = plt.subplots(figsize=(12, 5))

colors_shock = [RED if v < 0 else GREEN for v in covid_drop.values]

ax.bar(
    covid_drop.index,
    covid_drop.values,
    color=colors_shock,
    edgecolor='white'
)

# Baseline line
ax.axhline(0, color='black', lw=0.8)

# Labels & Title
ax.set_title(
    "COVID-19 Employment Shock: Change 2019→2020 (pp)\n(15-64, avg M+F)",
    fontweight='bold',
    fontsize=13
)
ax.set_ylabel("Percentage-point change")
ax.set_xlabel("Country")

plt.xticks(rotation=45, ha='right')

# ==============================
# 3. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig5_covid_shock.png


In [62]:
# ── FIG 6: Senior employment surge ───────────────────────────
import os
import matplotlib.pyplot as plt

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "fig6_senior_growth.png")

# ==============================
# 2. PREPARE DATA
# ==============================
senior_country = (
    emp_5564
    .groupby(['ISO', 'YEAR'])['RATE'].mean()
    .unstack()
    .dropna(subset=[2015, 2024])
)

senior_country['growth'] = senior_country[2024] - senior_country[2015]
sc = senior_country['growth'].sort_values(ascending=False)

# ==============================
# 3. CREATE PLOT
# ==============================
fig, ax = plt.subplots(figsize=(12, 5))

ax.bar(sc.index, sc.values, color=BLUE, edgecolor='white')

# Mean line
mean_growth = sc.mean()
ax.axhline(mean_growth, color=ORANGE, lw=1.5, linestyle='--',
           label=f'Mean growth = {mean_growth:.1f} pp')

# Labels & Title
ax.set_title(
    "Senior (55-64) Employment Growth 2015→2024 by Country (pp)",
    fontweight='bold',
    fontsize=13
)
ax.set_ylabel("Percentage-point change")
ax.set_xlabel("Country")

ax.legend()
plt.xticks(rotation=45, ha='right')

# ==============================
# 4. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig6_senior_growth.png


In [63]:
# ── FIG 7: Box plots by age group ────────────────────────────
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "fig7_boxplot_age.png")

# ==============================
# 2. CREATE PLOT
# ==============================
fig, ax = plt.subplots(figsize=(11, 6))

order = ['15-24', '15-29', '25-54', '55-64', '15-64', '20-64']

data_box = [
    df_long[df_long['AGE'] == a]['RATE'].dropna().values
    for a in order
]

bp = ax.boxplot(
    data_box,
    labels=order,
    patch_artist=True,
    medianprops=dict(color='black', lw=2)
)

# Apply colors
colors_box = sns.color_palette("pastel", len(order))
for patch, color in zip(bp['boxes'], colors_box):
    patch.set_facecolor(color)

# Labels & Title
ax.set_title(
    "Employment Rate Distribution by Age Group (2015–2024)",
    fontweight='bold',
    fontsize=13
)
ax.set_xlabel("Age Group")
ax.set_ylabel("Employment Rate (%)")

# ==============================
# 3. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig7_boxplot_age.png


In [64]:
# ── FIG 8: Country heatmap (15-64, avg) ──────────────────────
import os
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# 1. DEFINE OUTPUT DIRECTORY
# ==============================
output_dir = r"D:\Data Analytics\Project 5\Project 5.1\Output"
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(output_dir, "fig8_country_heatmap.png")

# ==============================
# 2. PREPARE DATA
# ==============================
heat_data = (
    emp_1564
    .groupby(['ISO', 'YEAR'])['RATE'].mean()
    .unstack()
    .dropna(how='all')
    .sort_values(2024, ascending=False)
)

# ==============================
# 3. CREATE HEATMAP
# ==============================
fig, ax = plt.subplots(figsize=(14, 10))

sns.heatmap(
    heat_data,
    annot=True,
    fmt=".1f",
    cmap="YlGn",
    linewidths=0.3,
    ax=ax,
    cbar_kws={'label': 'Employment Rate (%)'}
)

# Labels & Title
ax.set_title(
    "Employment Rate Heatmap by Country & Year (15-64, avg M+F)",
    fontweight='bold',
    fontsize=13
)
ax.set_xlabel("Year")
ax.set_ylabel("Country")

# ==============================
# 4. SAVE + SHOW (CORRECT ORDER)
# ==============================
plt.tight_layout()
plt.savefig(file_path, dpi=150, bbox_inches='tight')
plt.show()
plt.close()

print(f"✓ Figure saved at: {file_path}")

✓ Figure saved at: D:\Data Analytics\Project 5\Project 5.1\Output\fig8_country_heatmap.png


In [65]:
summary = pd.DataFrame({
    'Mean 2015': emp_1564[emp_1564['YEAR']==2015].groupby('ISO')['RATE'].mean().round(1),
    'Mean 2024': emp_1564[emp_1564['YEAR']==2024].groupby('ISO')['RATE'].mean().round(1),
    'Growth pp': total_growth.round(1),
    'Gap M-F 2024 pp': gap_2024['gap'].round(1),
}).dropna(subset=['Mean 2015','Mean 2024'])
print(summary.to_string())

     Mean 2015  Mean 2024  Growth pp  Gap M-F 2024 pp
ISO                                                  
AT        71.1       74.1        3.0              6.8
BE        61.8       66.8        5.0              6.9
BG        62.0       70.8        8.8              6.5
CH        79.2       80.4        1.2              6.7
CY        62.8       75.7       12.8              9.2
CZ        70.2       75.3        5.1             11.6
DE        73.0       77.4        4.4              6.6
DK        72.0       77.2        5.2              5.4
EE        72.1       75.7        3.6              1.0
ES        57.8       66.0        8.2              8.9
FI        67.6       72.6        5.0              0.4
FR        64.8       69.0        4.2              5.2
GR        50.8       63.3       12.5             17.4
HR        56.0       68.2       12.2              5.7
HU        65.9       75.1        9.2              7.3
IE        64.8       74.5        9.7              7.8
IS        84.7       85.2   